# Data Preprocessing & Feature Engineering

In [13]:
import pandas as pd

In [14]:
df = pd.read_csv('../data/Steel_industry_data.csv')
df.head()

,date,Usage_kWh,Lagging_Current_Reactive.Power_kVarh,Leading_Current_Reactive_Power_kVarh,CO2(tCO2),Lagging_Current_Power_Factor,Leading_Current_Power_Factor,NSM,WeekStatus,Day_of_week,Load_Type
0,01/01/2018 00:15,3.17,2.95,0.0,0.0,73.21,100.0,900,Weekday,Monday,Light_Load
1,01/01/2018 00:30,4.00,4.46,0.0,0.0,66.77,100.0,1800,Weekday,Monday,Light_Load
2,01/01/2018 00:45,3.24,3.28,0.0,0.0,70.28,100.0,2700,Weekday,Monday,Light_Load
3,01/01/2018 01:00,3.31,3.56,0.0,0.0,68.09,100.0,3600,Weekday,Monday,Light_Load
4,01/01/2018 01:15,3.82,4.50,0.0,0.0,64.72,100.0,4500,Weekday,Monday,Light_Load


## Chronological Ordering

In [15]:
# ordering date/time chronologically
df['date'] = pd.to_datetime(df['date'], format = '%d/%m/%Y %H:%M')
df = df.sort_values(by = 'date', ascending = True).reset_index(drop = True)
df.head()

,date,Usage_kWh,Lagging_Current_Reactive.Power_kVarh,Leading_Current_Reactive_Power_kVarh,CO2(tCO2),Lagging_Current_Power_Factor,Leading_Current_Power_Factor,NSM,WeekStatus,Day_of_week,Load_Type
0,2018-01-01 00:00:00,3.42,3.46,0.0,0.0,70.30,100.0,0,Weekday,Monday,Light_Load
1,2018-01-01 00:15:00,3.17,2.95,0.0,0.0,73.21,100.0,900,Weekday,Monday,Light_Load
2,2018-01-01 00:30:00,4.00,4.46,0.0,0.0,66.77,100.0,1800,Weekday,Monday,Light_Load
3,2018-01-01 00:45:00,3.24,3.28,0.0,0.0,70.28,100.0,2700,Weekday,Monday,Light_Load
4,2018-01-01 01:00:00,3.31,3.56,0.0,0.0,68.09,100.0,3600,Weekday,Monday,Light_Load


In [16]:
# checking the interval between date samples
df['date'].diff().unique()

<TimedeltaArray>
[NaT, '0 days 00:15:00']
Length: 2, dtype: timedelta64[us]

Timestamp Handling: The raw dataset stores the 00:00 observation at the end of each daily block, after the 23:45 observation. In preprocessing, timestamps are interpreted according to their recorded date and time, with 00:00 treated as the first 15-minute interval of the corresponding calendar day. The observations are therefore reordered chronologically before time-based and lag features are created.

## Initial Data Checks

In [17]:
# checking for missing values and duplicate rows
print("Missing values:", df.isna().sum().sum())
print("Duplicate rows:", df.duplicated().sum())

Missing values: 0
Duplicate rows: 0


## Time-Based Features

In [18]:
# creating time features
df['hour'] = df['date'].dt.hour
df['minute'] = df['date'].dt.minute
df['month'] = df['date'].dt.month

## Lag Features

In [19]:
# creating lag features (previous observations)
df['lag_1'] = df['Usage_kWh'].shift(1) # 15 minutes ago
df['lag_2'] = df['Usage_kWh'].shift(2) # 30 minutes ago
df['lag_4'] = df['Usage_kWh'].shift(4) # 1 hour ago
df['lag_96'] = df['Usage_kWh'].shift(96) # 1 day ago
df['lag_672'] = df['Usage_kWh'].shift(672) # 1 week ago

## Remove Rows Without Sufficient Lag History

In [20]:
df = df.drop(df.index[0:672]).reset_index(drop = True)

In [21]:
model_df = df[['date','hour','minute','month','Day_of_week','WeekStatus','lag_1','lag_2','lag_4',
    'lag_96','lag_672','Usage_kWh']]

## Chronological Train-Validation-Test Split

In [22]:
train_size = int(len(model_df)*0.7)
val_size = int(len(model_df)*0.15)

train = model_df.iloc[:train_size]
val = model_df.iloc[train_size:train_size+val_size]
test = model_df.iloc[train_size+val_size:]

In [23]:
print(train.shape)
print(val.shape)
print(test.shape)
print("Train:", train['date'].min(), "to", train['date'].max())
print("Validation:", val['date'].min(), "to", val['date'].max())
print("Test:", test['date'].min(), "to", test['date'].max())

(24057, 12)
(5155, 12)
(5156, 12)
Train: 2018-01-08 00:00:00 to 2018-09-15 14:00:00
Validation: 2018-09-15 14:15:00 to 2018-11-08 06:45:00
Test: 2018-11-08 07:00:00 to 2018-12-31 23:45:00


## Save Processed Datasets

In [24]:
train.to_csv('../data/train.csv', index=False)
val.to_csv('../data/validation.csv', index=False)
test.to_csv('../data/test.csv', index=False)
print("Processed datasets saved successfully.")

Processed datasets saved successfully.


Summary: The data preprocessing stage converted the timestamp variable to datetime format, ordered the observations chronologically, and confirmed the expected 15-minute sampling interval. Time-based features including hour, minute, and month were extracted, while lag features representing previous 15-minute, 30-minute, 1-hour, 1-day, and 1-week energy usage were created to support forecasting. The first 672 observations were removed because they lacked sufficient historical data for the weekly lag feature. The resulting dataset was divided chronologically into training, validation, and test sets using a 70/15/15 split, with the processed datasets saved for subsequent modelling. The zero-usage observation identified during EDA was retained because there was insufficient evidence to confirm that it was a measurement error.